# Network Optimisation

The baseline U-Net provides a first benchmark for the solar filament segmentation. On the local validation and test sets, the model achieved PQ scores of 0.327 and 0.292, respectively.

This notebook focuses on improving the segmentation pipeline by systematically evaluating different training configurations and network architectures.

The experiments will be performed incrementally, keeping the data split and evaluation protocol fixed to ensure a fair comparison between configurations.

The optimisation process will first focus on the existing U-Net architecture and training strategy. Once a strong U-Net baseline has been established, alternative architectures will be evaluated.

## Imports

In [ ]:
from pathlib import Path
import sys
import random
from sklearn.model_selection import ParameterSampler
import torch

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import get_project_root, load_config, get_path
from src.utils.data import (
    get_train_images_dir,
    get_test_images_dir,
    get_annotations_path,
    load_annotations,
    remove_duplicate_annotations
)
from src.processing.masks import build_masks
from src.splitting.split import split_masks
from src.datasets.segmentation import SegmentationDataset, InferenceDataset
from src.training.dataloaders import create_dataloaders
from src.models.unet import UNet, ConfigurableUNet
from src.training.losses import BCEDiceLoss
from src.training.trainer import train_model
from src.processing.masks import get_connected_components, calculate_iou, filter_components_by_area
from src.utils.submission import (
    masks_to_submission,
    save_submission,
)

PROJECT_ROOT = get_project_root()
CONFIG = load_config()

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\Darío\Desktop\KAGGLE\kaggle_solar_filament_segmentation


## Load data

In [3]:
RAW_DATA_DIR = PROJECT_ROOT / CONFIG["paths"]["raw_data_dir"]

TRAIN_IMAGES_DIR = get_train_images_dir(RAW_DATA_DIR)
TEST_IMAGES_DIR = get_test_images_dir(RAW_DATA_DIR)
ANNOTATIONS_PATH = get_annotations_path(RAW_DATA_DIR)

annotations = load_annotations(ANNOTATIONS_PATH)

annotations_clean = remove_duplicate_annotations(annotations)

print(f"Images: {len(annotations_clean['images']):,}")
print(f"Annotations: {len(annotations_clean['annotations']):,}")

Images: 707
Annotations: 5,209


## Build masks

In [4]:
final_masks = build_masks(annotations_clean)

file_name = random.choice(list(final_masks.keys()))

mask = final_masks[file_name][0]

print(mask)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


## Splitting

### Data splitting strategy

Given the high computational cost of training segmentation models on **2048 × 2048 images**, performing cross-validation for every experiment would substantially increase the computational budget. Therefore, during the model optimization stage, we use a fixed **80/10/10 random split** consisting of 565 training, 71 validation, and 71 test images.

We also investigated the temporal distribution of the observations to assess whether a random split could introduce significant temporal leakage. After removing duplicated file names, most observations were temporally isolated, with **460 out of 571 temporal clusters containing a single image**. Therefore, no strong temporal grouping structure was identified that would justify a more restrictive temporal split.

The same fixed partitions will be used throughout the experiments to ensure a fair comparison between configurations. **Cross-validation may be applied afterwards to the final selected model** to obtain a more robust estimate of its performance.

In [16]:
train_masks, val_masks, test_masks = split_masks(
    final_masks,
    train_size=0.8,
    val_size=0.1,
    test_size=0.1,
    random_state=42,
)

train_dataset = SegmentationDataset(
    masks=train_masks,
    images_dir=TRAIN_IMAGES_DIR,
)

val_dataset = SegmentationDataset(
    masks=val_masks,
    images_dir=TRAIN_IMAGES_DIR,
)

test_dataset = SegmentationDataset(
    masks=test_masks,
    images_dir=TRAIN_IMAGES_DIR,
)

train_loader, val_loader, test_loader = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_dataset=test_dataset,
    batch_size=1,
    num_workers=0,
    pin_memory=True,
)

print(f"Train:      {len(train_masks):,} images")
print(f"Validation: {len(val_masks):,} images")
print(f"Test:       {len(test_masks):,} images")
print(f"Total:      {len(final_masks):,} images")

Train:      565 images
Validation: 71 images
Test:       71 images
Total:      707 images


## Architecture hyperparameter optimization

The "ConfigurableUNet" allow us to investigate how the network architecture affects segmentation performance. Instead of exhaustively evaluating all possible combinations, we use a Random Search strategy to sample a limited number of configurations from the predefined parameter space.

In [10]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    
param_space = {
    "base_channels": [8, 16, 32],
    "depth": [3, 4, 5],
    "num_convs": [1, 2, 3],
    "kernel_size": [3, 5],
    "activation": [
        "relu",
        "leaky_relu",
        "gelu",
        "silu",
    ],
}

TRAINING_PARAMS = {
    "learning_rate": 1e-3,
    "weight_decay": 0.0,
    "patience": 7,
    "scheduler_patience": 3,
    "scheduler_factor": 0.5,
    "threshold": 0.5,
    "device": device,
    "use_amp": True,
}

N_ITER = 15
RANDOM_STATE = 42

experiments = list(
    ParameterSampler(
        param_distributions=param_space,
        n_iter=N_ITER,
        random_state=RANDOM_STATE,
    )
)

print(f"Total possible configurations: 216")
print(f"Configurations sampled: {len(experiments)}")

Device: cuda
GPU: NVIDIA GeForce RTX 4070
Total possible configurations: 216
Configurations sampled: 15


In [11]:
loss_fn = BCEDiceLoss(
    bce_weight=0.5,
    dice_weight=0.5,
)

In [ ]:
results = []

for experiment_id, params in enumerate(
    experiments,
    start=1,
):

    print("\n" + "=" * 80)
    print(
        f"Experiment {experiment_id}/{N_ITER}"
    )
    print(params)
    print("=" * 80)

    model = ConfigurableUNet(
        **params,
    )

    models_dir = get_path(CONFIG["paths"]["models_dir"])

    checkpoint_path = (
        Path("artifacts/models")
        / f"architecture_search_{experiment_id:03d}.pth"
    )

    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        loss_fn=loss_fn,
        checkpoint_path=checkpoint_path,
        **TRAINING_PARAMS,
    )

    best_epoch = (
        history["val_dice"].index(
            max(history["val_dice"])
        )
        + 1
    )

    best_val_dice = max(
        history["val_dice"]
    )

    best_val_iou = max(
        history["val_iou"]
    )

    results.append(
        {
            "experiment": experiment_id,
            **params,
            "best_epoch": best_epoch,
            "best_val_dice": best_val_dice,
            "best_val_iou": best_val_iou,
            "checkpoint": str(
                checkpoint_path
            ),
        }
    )


Experiment 1/15
{'num_convs': 1, 'kernel_size': 5, 'depth': 3, 'base_channels': 32, 'activation': 'silu'}
Epoch 01/50 | Train Loss: 0.4023 | Train Dice: 0.3729 | Train IoU: 0.2514 | Val Loss: 0.2689 | Val Dice: 0.5004 | Val IoU: 0.3469 | LR: 1.00e-03
  -> Best model saved (Val Dice: 0.5004)
Epoch 02/50 | Train Loss: 0.2530 | Train Dice: 0.5239 | Train IoU: 0.3677 | Val Loss: 0.3006 | Val Dice: 0.4281 | Val IoU: 0.2863 | LR: 1.00e-03
Epoch 03/50 | Train Loss: 0.2367 | Train Dice: 0.5530 | Train IoU: 0.3942 | Val Loss: 0.3465 | Val Dice: 0.3617 | Val IoU: 0.2312 | LR: 1.00e-03
Epoch 04/50 | Train Loss: 0.2349 | Train Dice: 0.5556 | Train IoU: 0.3966 | Val Loss: 0.2362 | Val Dice: 0.5487 | Val IoU: 0.3959 | LR: 1.00e-03
  -> Best model saved (Val Dice: 0.5487)
Epoch 05/50 | Train Loss: 0.2180 | Train Dice: 0.5876 | Train IoU: 0.4269 | Val Loss: 0.2423 | Val Dice: 0.5383 | Val IoU: 0.3860 | LR: 1.00e-03
Epoch 06/50 | Train Loss: 0.2160 | Train Dice: 0.5914 | Train IoU: 0.4300 | Val Loss: 